# ELEC378 Final Project
Authors: Rocky Ren, Harvey Chen, Matthew Karazincir

In [3]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import sklearn
from sklearn.model_selection import train_test_split
import tqdm

In [4]:
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "butterfly_data")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
CSV_PATH = os.path.join(DATA_DIR, "train.csv")

RANDOM_STATE = 42

In [5]:
def load_metadata():
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df)} entries, {df['TARGET'].nunique()} classes")
    return df

def load_image(filename, size):
    path = os.path.join(TRAIN_IMG_DIR, filename)
    img = Image.open(path).convert("RGB").resize((size, size))
    return np.array(img)

def load_image_label_pair(df, index, size=224):
    row = df.iloc[index]
    img = load_image(row["file_name"], size)
    label = row["TARGET"]
    return img, label

In [6]:
df = load_metadata()
#df.head()

df["TARGET"].nunique()


Loaded 12594 entries, 100 classes


100

In [7]:
img, label = load_image_label_pair(df, 0)
print(f"Sample pair — label: {label}, image shape: {img.shape}")

Sample pair — label: ADONIS, image shape: (224, 224, 3)


In [8]:
X_train_files, X_val_files, y_train, y_val = train_test_split(
    df["file_name"].values,
    df["TARGET"].values,
    test_size=0.2,
    stratify=df["TARGET"].values,
    random_state=RANDOM_STATE,
)
print(f"Train: {len(X_train_files)}, Validation: {len(X_val_files)}")

Train: 10075, Validation: 2519


In [ ]:
# X_train = []
# for file in X_train_files:
#     img = load_image(file, 224)
#     X_train.append(img.flatten())
#
# X_val = []
# for file in X_val_files:
#     img = load_image(file, 224)
#     X_val.append(img.flatten())

In [ ]:
# # Scree plot
# svd = sklearn.decomposition.TruncatedSVD(n_components=128)
# X_reduced = svd.fit_transform(X_train)
#
# plt.plot(svd.explained_variance_ratio_)

In [ ]:
# # PCA
#
# pca = sklearn.decomposition.PCA(n_components=101)
# X_train_pca = pca.fit_transform(X_train)
# X_val_pca = pca.transform(X_val)
#
# lda = sklearn.discriminant_analysis.LinearDiscriminantAnalysis(n_components=100)
# X_train_lda = lda.fit_transform(X_train_pca, y_train)
# X_val_lda = lda.transform(X_val_pca)

In [ ]:
# model = sklearn.linear_model.LogisticRegression()
# model.fit(X_train_lda, y_train)

In [ ]:
# predictions = model.predict(X_val_lda)
# sklearn.metrics.accuracy_score(y_val, predictions)

## SIFT Bag-of-Words pipeline
Replace raw flattened pixels with SIFT descriptors aggregated into a visual-word histogram per image.

In [ ]:
# import cv2
#
# def extract_sift(filename, size=224):
#     img = load_image(filename, size)
#     gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
#     sift = cv2.SIFT_create()
#     _, descs = sift.detectAndCompute(gray, None)
#     return descs if descs is not None else np.zeros((0, 128), dtype=np.float32)
#
# train_descs = [extract_sift(f) for f in tqdm.tqdm(X_train_files, desc="SIFT train")]
# val_descs   = [extract_sift(f) for f in tqdm.tqdm(X_val_files,   desc="SIFT val")]
#
# print(f"avg keypoints/image: {np.mean([len(d) for d in train_descs]):.0f}")

In [ ]:
# from sklearn.cluster import MiniBatchKMeans
#
# K = 256
# all_descs = np.vstack([d for d in train_descs if len(d) > 0])
# rng = np.random.RandomState(RANDOM_STATE)
# sample = all_descs[rng.choice(len(all_descs), min(100_000, len(all_descs)), replace=False)]
#
# kmeans = MiniBatchKMeans(n_clusters=K, batch_size=1000, random_state=RANDOM_STATE, n_init=3)
# kmeans.fit(sample)
#
# def to_hist(descs):
#     if len(descs) == 0:
#         return np.zeros(K, dtype=np.float32)
#     labels = kmeans.predict(descs)
#     h = np.bincount(labels, minlength=K).astype(np.float32)
#     return h / (h.sum() + 1e-12)
#
# X_train_sift = np.vstack([to_hist(d) for d in train_descs])
# X_val_sift   = np.vstack([to_hist(d) for d in val_descs])
# print("SIFT-BoW shapes:", X_train_sift.shape, X_val_sift.shape)

In [ ]:
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score
#
# lda = LinearDiscriminantAnalysis(n_components=99)
# X_train_lda_sift = lda.fit_transform(X_train_sift, y_train)
# X_val_lda_sift   = lda.transform(X_val_sift)
#
# model = LogisticRegression(max_iter=1000)
# model.fit(X_train_lda_sift, y_train)
# acc = accuracy_score(y_val, model.predict(X_val_lda_sift))
# print(f"SIFT-BoW + LDA + LogReg accuracy: {acc:.4f}")

# NEURAL NETWORK

In [64]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets

In [65]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
# class NeuralNetwork(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.flatten = nn.Flatten()
#         self.linear_relu_stack = nn.Sequential(
#             nn.Linear(28*28, 512),
#             nn.ReLU(),
#             nn.Linear(512, 512),
#             nn.ReLU(),
#             nn.Linear(512, 10),
#         )
#
#     def forward(self, x):
#         x = self.flatten(x)
#         logits = self.linear_relu_stack(x)
#         return logits

In [ ]:
# model = NeuralNetwork().to(device)
# print(model)

In [ ]:
# X = torch.rand(1, 28, 28, device=device)
# logits = model(X)
# pred_probab = nn.Softmax(dim=1)(logits)
# y_pred = pred_probab.argmax(1)
# print(f"Predicted class: {y_pred}")
# print(pred_probab)

In [ ]:
# input_image = torch.rand(3,28,28)
# print(input_image.size())

In [ ]:
# flatten = nn.Flatten()
# flat_image = flatten(input_image)
# print(flat_image.size())

In [ ]:
# layer1 = nn.Linear(in_features=28*28, out_features=20)
# hidden1 = layer1(flat_image)
# print(hidden1.size())

In [ ]:
# print(f"Before ReLU: {hidden1}\n\n")
# hidden1 = nn.ReLU()(hidden1)
# print(f"After ReLU: {hidden1}")

In [ ]:
# seq_modules = nn.Sequential(
#     flatten,
#     layer1,
#     nn.ReLU(),
#     nn.Linear(20, 10)
# )
# input_image = torch.rand(3,28,28)
# logits = seq_modules(input_image)

In [ ]:
# softmax = nn.Softmax(dim=1)
# pred_probab = softmax(logits)
# print(pred_probab)

In [ ]:
# print(f"Model structure: {model}\n\n")
#
# for name, param in model.named_parameters():
#     print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

In [ ]:
# class CNN(nn.Module):
#     def __init__(self, num_classes=100, p_drop=0.15, p_drop_fc=0.5):
#         super().__init__()
#
#         def block(in_c, out_c):
#             return nn.Sequential(
#                 nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
#                 nn.BatchNorm2d(out_c),
#                 nn.ReLU(inplace=True),
#                 nn.MaxPool2d(2),
#                 nn.Dropout2d(p_drop),
#             )
#
#         self.features = nn.Sequential(
#             block(3,   32),    # 224 -> 112
#             block(32,  64),    # 112 -> 56
#             block(64,  128),   # 56  -> 28
#             block(128, 256),   # 28  -> 14
#             block(256, 512),   # 14  -> 7
#         )
#         self.gap = nn.AdaptiveAvgPool2d(1)
#         self.classifier = nn.Sequential(
#             nn.Linear(512, 256),
#             nn.ReLU(inplace=True),
#             nn.Dropout(p_drop_fc),
#             nn.Linear(256, num_classes),
#         )
#
#     def forward(self, x):
#         x = self.features(x)        # (B, 512, 7, 7)
#         x = self.gap(x)             # (B, 512, 1, 1)
#         x = torch.flatten(x, 1)     # (B, 512)
#         return self.classifier(x)   # (B, num_classes)

# ResNet-18 
model = models.resnet18(weights=None, num_classes=100).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Total parameters: {n_params:,}")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import WeightedRandomSampler

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc   = label_encoder.transform(y_val)

IMG_SIZE = 224
BATCH_SIZE = 64

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

class ButterflyDataset(Dataset):
    def __init__(self, files, labels, transform):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(TRAIN_IMG_DIR, self.files[idx])).convert("RGB")
        return self.transform(img), int(self.labels[idx])

train_ds = ButterflyDataset(X_train_files, y_train_enc, train_tf)
val_ds   = ButterflyDataset(X_val_files,   y_val_enc,   val_tf)

# class-balanced sampling: each class gets equal expected weight per epoch.
class_counts = np.bincount(y_train_enc)
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_train_enc]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(y_train_enc),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=4, pin_memory=True)

print(f"Class counts — min: {class_counts.min()}, max: {class_counts.max()}, mean: {class_counts.mean():.1f}")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)  # lr from 3-fold CV (mean val_acc 0.5291)

EPOCHS = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

MIXUP_ALPHA = 0.2

def mixup(x, y, alpha):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[perm]
    return x_mix, y, y[perm], lam

best_val = 0.0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        if MIXUP_ALPHA > 0:
            x_mix, ya, yb, lam = mixup(imgs, labels, MIXUP_ALPHA)
            logits = model(x_mix)
            loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        # accuracy on original labels (mixup makes this an underestimate, but useful as a trend)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
        pbar.set_postfix(
            loss=f"{running_loss/total:.4f}",
            acc=f"{correct/total:.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.5f}",
        )

    scheduler.step()

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for imgs, labels in tqdm.tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]"):
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            logits = model(imgs)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total += imgs.size(0)
    val_acc = val_correct / val_total
    best_val = max(best_val, val_acc)
    print(f"Epoch {epoch+1}: train_acc={correct/total:.4f}  val_acc={val_acc:.4f}  best={best_val:.4f}")

## Patch-based dictionary learning (no SIFT)
Sample random raw image patches, learn a sparse-coding dictionary directly on them, encode every dense patch in each image, then spatial-max-pool the codes 

In [ ]:
from sklearn.decomposition import MiniBatchDictionaryLearning
from numpy.lib.stride_tricks import sliding_window_view

IMG_SIZE_DICT = 64        # downscale images for the patch pipeline
PATCH_SIZE   = 8          # 8x8 patches
STRIDE       = 2          # dense extraction with stride 2
N_ATOMS      = 256        # dictionary size
ALPHA        = 1.0        # sparsity penalty during dictionary fit
ENC_ALPHA    = 0.1        # threshold for fast encoding at inference
N_FIT_PATCHES = 50_000    # patches used to fit the dictionary

def load_gray(filename, size=IMG_SIZE_DICT):
    img = Image.open(os.path.join(TRAIN_IMG_DIR, filename)).convert("L").resize((size, size))
    return np.array(img, dtype=np.float32) / 255.0

print("Loading images at low resolution...")
train_imgs = np.stack([load_gray(f) for f in tqdm.tqdm(X_train_files, desc="train")])
val_imgs   = np.stack([load_gray(f) for f in tqdm.tqdm(X_val_files,   desc="val")])
print("Image arrays:", train_imgs.shape, val_imgs.shape)

def dense_patches(img):
    """Return all PATCH_SIZE x PATCH_SIZE patches as (Hp, Wp, P*P)."""
    w = sliding_window_view(img, (PATCH_SIZE, PATCH_SIZE))[::STRIDE, ::STRIDE]
    return w.reshape(*w.shape[:2], -1)

def normalize_patches(P):
    """Per-patch contrast normalization: subtract mean, divide by std."""
    P = P - P.mean(axis=-1, keepdims=True)
    s = P.std(axis=-1, keepdims=True)
    return P / np.maximum(s, 1e-6)

# Sample patches for dictionary fitting
rng = np.random.RandomState(RANDOM_STATE)
patches_per_img = max(1, N_FIT_PATCHES // len(train_imgs) + 1)
print(f"Sampling ~{patches_per_img} patches per image for dictionary learning...")
fit_patches = []
for img in tqdm.tqdm(train_imgs, desc="sample"):
    P = dense_patches(img).reshape(-1, PATCH_SIZE * PATCH_SIZE)
    idx = rng.choice(len(P), size=min(patches_per_img, len(P)), replace=False)
    fit_patches.append(P[idx])
fit_patches = np.vstack(fit_patches)[:N_FIT_PATCHES].astype(np.float32)
fit_patches = normalize_patches(fit_patches)
print(f"Fitting dictionary on {len(fit_patches)} patches of dim {fit_patches.shape[1]}...")

dico = MiniBatchDictionaryLearning(
    n_components=N_ATOMS,
    alpha=ALPHA,
    batch_size=256,
    max_iter=30,
    random_state=RANDOM_STATE,
    transform_algorithm="threshold",   # fast soft-threshold encoding at inference
    transform_alpha=ENC_ALPHA,
)
dico.fit(fit_patches)
print(f"Dictionary shape: {dico.components_.shape}")


fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for ax, atom in zip(axes.flat, dico.components_[:64]):
    ax.imshow(atom.reshape(PATCH_SIZE, PATCH_SIZE), cmap="gray")
    ax.axis("off")
plt.suptitle("Learned dictionary atoms (first 64)")
plt.show()

In [ ]:
def encode_image(img):
    """Dense-encode all patches in one image, then 2x2 spatial-max-pool the codes."""
    P = dense_patches(img)                                 # (Hp, Wp, P*P)
    Hp, Wp = P.shape[:2]
    flat = normalize_patches(P.reshape(-1, P.shape[-1]))   # (Hp*Wp, P*P)
    codes = dico.transform(flat)                            # (Hp*Wp, N_ATOMS)
    codes = codes.reshape(Hp, Wp, N_ATOMS)
    h, w = Hp // 2, Wp // 2
    quads = np.stack([
        codes[:h,  :w ].max(axis=(0, 1)),
        codes[:h,  w: ].max(axis=(0, 1)),
        codes[h:,  :w ].max(axis=(0, 1)),
        codes[h:,  w: ].max(axis=(0, 1)),
    ])
    return quads.flatten()                                  # 4 * N_ATOMS

print("Encoding train images...")
X_train_dl = np.vstack([encode_image(img) for img in tqdm.tqdm(train_imgs, desc="encode train")])
print("Encoding val images...")
X_val_dl   = np.vstack([encode_image(img) for img in tqdm.tqdm(val_imgs,   desc="encode val")])
print("Feature shapes:", X_train_dl.shape, X_val_dl.shape)

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

clf = SVC(kernel="rbf", C=10, gamma="scale")
print("Fitting RBF SVM...")
clf.fit(X_train_dl, y_train)
acc = accuracy_score(y_val, clf.predict(X_val_dl))
print(f"Patch dictionary learning + 2x2 max-pool + RBF SVM: {acc:.4f}")

## K-fold CV for ResNet hyperparameter tuning


In [ ]:
from sklearn.model_selection import StratifiedKFold
import gc

# --- Search grid 
LR_GRID    = [3e-4, 1e-3, 3e-3]
K_FOLDS    = 3
CV_EPOCHS  = 4
WD         = 1e-4
LSMOOTH    = 0.1
CV_MIXUP_A = 0.2

def cv_mixup(x, y, alpha):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

def train_eval_fold(tr_files, tr_labels, va_files, va_labels, lr, epochs):
    m = models.resnet18(weights=None, num_classes=100).to(device)
    opt = optim.AdamW(m.parameters(), lr=lr, weight_decay=WD)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=LSMOOTH)

    tr_ds = ButterflyDataset(tr_files, tr_labels, train_tf)
    va_ds = ButterflyDataset(va_files, va_labels, val_tf)

    cnts = np.bincount(tr_labels, minlength=100)
    sw = (1.0 / np.maximum(cnts, 1))[tr_labels]
    samp = WeightedRandomSampler(weights=sw, num_samples=len(tr_labels), replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=samp, num_workers=4, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,   num_workers=4, pin_memory=True)

    for epoch in range(epochs):
        m.train()
        for imgs, labs in tqdm.tqdm(tr_loader, desc=f"    ep{epoch+1}/{epochs}", leave=False):
            imgs, labs = imgs.to(device, non_blocking=True), labs.to(device, non_blocking=True)
            opt.zero_grad()
            x_mix, ya, yb, lam = cv_mixup(imgs, labs, CV_MIXUP_A)
            logits = m(x_mix)
            loss = lam * crit(logits, ya) + (1 - lam) * crit(logits, yb)
            loss.backward()
            opt.step()
        sch.step()

    m.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labs in va_loader:
            imgs, labs = imgs.to(device, non_blocking=True), labs.to(device, non_blocking=True)
            correct += (m(imgs).argmax(1) == labs).sum().item()
            total += imgs.size(0)

    del m, opt, sch, crit, tr_loader, va_loader, samp
    gc.collect()
    torch.cuda.empty_cache()
    return correct / total

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print(f"{K_FOLDS}-fold CV across {len(LR_GRID)} LRs, {CV_EPOCHS} epochs/fold")
print(f"Total: {K_FOLDS * len(LR_GRID) * CV_EPOCHS} epoch-trainings\n")

for lr in LR_GRID:
    print(f"=== LR = {lr:.1e} ===")
    fold_accs = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_files, y_train_enc)):
        print(f"  Fold {fold+1}/{K_FOLDS}")
        acc = train_eval_fold(
            X_train_files[tr_idx], y_train_enc[tr_idx],
            X_train_files[va_idx], y_train_enc[va_idx],
            lr=lr, epochs=CV_EPOCHS,
        )
        fold_accs.append(acc)
        print(f"    fold val_acc = {acc:.4f}")
    mean_acc = float(np.mean(fold_accs))
    cv_results[lr] = (mean_acc, fold_accs)
    print(f"  LR={lr:.1e}  mean={mean_acc:.4f}  folds=[{', '.join(f'{a:.4f}' for a in fold_accs)}]\n")


for lr, (mean, accs) in sorted(cv_results.items()):
    print(f"  LR={lr:.1e}  mean={mean:.4f}  std={np.std(accs):.4f}")
best_lr = max(cv_results, key=lambda k: cv_results[k][0])
print(f"\nBest LR: {best_lr:.1e}  (mean CV val_acc = {cv_results[best_lr][0]:.4f})")

# Variational Autoencoder (generative model)


In [ ]:
import torch.nn.functional as F

VAE_IMG_SIZE = 64
LATENT_DIM = 128

class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()

        # Encoder: 64 -> 32 -> 16 -> 8 -> 4
        self.encoder = nn.Sequential(
            nn.Conv2d(3,   32,  kernel_size=4, stride=2, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32,  64,  kernel_size=4, stride=2, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64,  128, kernel_size=4, stride=2, padding=1),  nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),  nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.flatten_dim = 256 * 4 * 4
        self.fc_mu     = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)

        # Decoder: 4 -> 8 -> 16 -> 32 -> 64
        self.fc_dec = nn.Linear(latent_dim, self.flatten_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64,  kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64,  32,  kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32,  3,   kernel_size=4, stride=2, padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 256, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(recon, x, mu, logvar, beta=1.0):
    # reconstruction 
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
    # KL divergence between N(mu, sigma^2) and N(0, I), summed over latent dims, averaged over batch
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon_loss + beta * kl, recon_loss, kl


vae = VAE().to(device)
n_params = sum(p.numel() for p in vae.parameters())
print(vae)
print(f"VAE parameters: {n_params:,}")

In [ ]:
vae_tf = transforms.Compose([
    transforms.Resize((VAE_IMG_SIZE, VAE_IMG_SIZE)),
    transforms.ToTensor(),
])
vae_train_ds = ButterflyDataset(X_train_files, y_train_enc, vae_tf)
vae_loader   = DataLoader(vae_train_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)

vae_optim = optim.Adam(vae.parameters(), lr=1e-3)
VAE_EPOCHS = 20
BETA = 1.0    # KL weight; smaller -> sharper recon, less generative; larger -> smoother prior, blurrier recon

for epoch in range(VAE_EPOCHS):
    vae.train()
    total_loss = total_recon = total_kl = 0.0
    n_seen = 0
    pbar = tqdm.tqdm(vae_loader, desc=f"VAE epoch {epoch+1}/{VAE_EPOCHS}")
    for imgs, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        vae_optim.zero_grad()
        recon, mu, logvar = vae(imgs)
        loss, rl, kl = vae_loss(recon, imgs, mu, logvar, beta=BETA)
        loss.backward()
        vae_optim.step()

        bs = imgs.size(0)
        total_loss  += loss.item() * bs
        total_recon += rl.item()   * bs
        total_kl    += kl.item()   * bs
        n_seen += bs
        pbar.set_postfix(
            loss=f"{total_loss/n_seen:.2f}",
            recon=f"{total_recon/n_seen:.2f}",
            kl=f"{total_kl/n_seen:.2f}",
        )
    print(f"Epoch {epoch+1}: total={total_loss/n_seen:.2f}  recon={total_recon/n_seen:.2f}  KL={total_kl/n_seen:.2f}")

In [ ]:
vae.eval()

# 1) Reconstructions: original vs decoded, on a handful of training images
with torch.no_grad():
    sample_imgs, _ = next(iter(vae_loader))
    sample_imgs = sample_imgs[:8].to(device)
    recon, _, _ = vae(sample_imgs)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(sample_imgs[i].cpu().permute(1, 2, 0).numpy())
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0).numpy())
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("orig", rotation=0, labelpad=30)
axes[1, 0].set_ylabel("recon", rotation=0, labelpad=30)
plt.suptitle("VAE reconstructions")
plt.tight_layout()
plt.show()

# 2) Random samples from prior N(0, I)
with torch.no_grad():
    z = torch.randn(8, LATENT_DIM, device=device)
    samples = vae.decode(z)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    axes[i].imshow(samples[i].cpu().permute(1, 2, 0).numpy())
    axes[i].axis("off")
plt.suptitle("Random samples from N(0, I)")
plt.tight_layout()
plt.show()

# 3) Latent interpolation between two real images: smooth morph if the latent space is well-shaped
with torch.no_grad():
    a, b = sample_imgs[0:1], sample_imgs[1:2]
    mu_a, _ = vae.encode(a)
    mu_b, _ = vae.encode(b)
    alphas = torch.linspace(0, 1, 8, device=device)
    interp_z = torch.stack([(1 - t) * mu_a[0] + t * mu_b[0] for t in alphas])
    interp = vae.decode(interp_z)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    axes[i].imshow(interp[i].cpu().permute(1, 2, 0).numpy())
    axes[i].axis("off")
plt.suptitle("Latent interpolation: image_0  →  image_1")
plt.tight_layout()
plt.show()